# Task 11: Production Optimization

Optimize BERT for inference: TorchScript, ONNX, Quantization.

In [ ]:
import torch
import torch.nn as nn
import numpy as np

torch.manual_seed(42)

## 1. TorchScript

Compile model to serialized format for faster execution.

In [ ]:
# Define model
class SimpleBERT(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(30000, 128)
        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=128, nhead=4, dim_feedforward=256, batch_first=True),
            num_layers=2
        )
        self.classifier = nn.Linear(128, 2)
        
    def forward(self, x):
        x = self.embed(x)
        x = self.encoder(x)
        return self.classifier(x[:, 0])

model = SimpleBERT()
model.eval()

# TorchScript trace
x = torch.randint(0, 30000, (1, 64))
traced_model = torch.jit.trace(model, x)
traced_model.save('bert_traced.pt')

print("TorchScript model saved!")
print(f"Original size: {sum(p.numel() for p in model.parameters()):,}")

## 2. ONNX Export

Export to ONNX for cross-platform deployment.

In [ ]:
# ONNX export
torch.onnx.export(
    model,
    x,
    'bert_model.onnx',
    input_names=['input_ids'],
    output_names=['logits'],
    dynamic_axes={'input_ids': {0: 'batch', 1: 'seq_len'}, 'logits': {0: 'batch'}},
    opset_version=14
)

import os
onnx_size = os.path.getsize('bert_model.onnx') / (1024*1024)
print(f"ONNX model saved! Size: {onnx_size:.2f} MB")

## 3. Quantization

Reduce model size and improve speed with INT8 quantization.

In [ ]:
# Dynamic quantization (simplest)
quantized_model = torch.quantization.quantize_dynamic(
    model,
    {nn.Embedding, nn.Linear},
    dtype=torch.qint8
)

# Save quantized model
torch.save(quantized_model.state_dict(), 'bert_quantized.pt')

# Compare sizes
import os
original_size = sum(p.numel() * p.element_size() for p in model.parameters()) / (1024*1024)
quantized_size = os.path.getsize('bert_quantized.pt') / (1024*1024)

print(f"Original model: {original_size:.2f} MB")
print(f"Quantized model: {quantized_size:.2f} MB")
print(f"Compression: {original_size/quantized_size:.2f}x")

In [ ]:
# Benchmark inference speed
import time

def benchmark(model, input_tensor, num_runs=100):
    model.eval()
    times = []
    
    with torch.no_grad():
        for _ in range(num_runs):
            start = time.perf_counter()
            _ = model(input_tensor)
            times.append(time.perf_counter() - start)
    
    return np.mean(times) * 1000  # ms

x = torch.randint(0, 30000, (32, 128))

# Original
original_time = benchmark(model, x)

# Traced
traced = torch.jit.load('bert_traced.pt')
traced_time = benchmark(traced, x)

print(f"Original: {original_time:.2f} ms")
print(f"TorchScript: {traced_time:.2f} ms")
print(f"Speedup: {original_time/traced_time:.2f}x")

## Summary
- ✅ TorchScript for faster inference
- ✅ ONNX for cross-platform deployment
- ✅ Quantization for model compression
- ✅ Benchmarking tools

## Next: [Task 12: Advanced Exercises](../task-12-advanced-exercises/overview.html)